# Verify exported encoder engines against the deployment pipeline

Deployment (per `client/client/codeword_subscriber.py` + `client_cpp/src/codeword_publisher.cpp`):
* Robot: C++ runs a TRT engine `enc.trt` -> token grid.
* Server: Python loads `decoder.pkl` (original DALL-E) -> reconstruction.

For each candidate encoder artifact this notebook:
1. Reads the engine/ONNX input shape and runs real test images through it.
2. Extracts a token grid (argmax over logits if output is 4D, identity if 3D).
3. Compares tokens to the PyTorch reference encoder at the same resolution.
4. Decodes the tokens with the deployment `decoder.pkl` and reports MSE/PSNR vs the original image.

If a freshly exported encoder produces near-random tokens (low agreement, low PSNR), the export is broken upstream regardless of how the int8 calibration looks.

In [ ]:
import os, sys
sys.path.insert(0, '/home/leonard/arc_ws/src')

from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import onnx
import onnxruntime as ort
import tensorrt as trt

from dall_e import load_model, map_pixels, unmap_pixels

device = torch.device('cuda')
torch.set_grad_enabled(False)
TRT_LOGGER = trt.Logger(trt.Logger.WARNING)
VOCAB = 8192

TEST_DIR = Path('/home/leonard/dvae/data/test')

# Edit these to point at whichever exported artifacts you want to verify.
ENC_CANDIDATES = [
    '/home/leonard/arc_ws/src/enc.onnx',
    '/home/leonard/arc_ws/src/enc.trt',
    '/home/leonard/arc_ws/src/enc.int8.trt',
]

N_EVAL = 8  # number of test images to evaluate on

In [2]:
def load_image(path: Path, H: int, W: int) -> torch.Tensor:
    img = Image.open(path).convert('RGB')
    short = min(H, W)
    s = min(img.size)
    if s < short:
        r = short / s
        img = TF.resize(img, (round(r * img.size[1]), round(r * img.size[0])),
                        interpolation=Image.LANCZOS)
    iw, ih = img.size
    if ih < H or iw < W:
        img = TF.resize(img, (max(ih, H), max(iw, W)),
                        interpolation=Image.LANCZOS)
        iw, ih = img.size
    img = TF.crop(img, max(0, (ih - H) // 2), max(0, (iw - W) // 2), H, W)
    return T.ToTensor()(img)

def load_test_batch(n: int, H: int, W: int) -> torch.Tensor:
    files = sorted(p for p in TEST_DIR.iterdir() if p.suffix.lower() == '.jpg')[:n]
    if len(files) < n:
        raise RuntimeError(f'only {len(files)} .jpg files in {TEST_DIR}, need {n}')
    return torch.stack([load_image(p, H, W) for p in files]).to(device)

def mse_psnr(a: torch.Tensor, b: torch.Tensor):
    err = a.float() - b.float()
    mse = (err ** 2).mean().item()
    psnr = 10 * np.log10(1.0 / max(mse, 1e-12))
    return mse, psnr

In [ ]:
_TRT2T = {
    trt.DataType.FLOAT: torch.float32,
    trt.DataType.HALF:  torch.float16,
    trt.DataType.INT32: torch.int32,
    trt.DataType.INT8:  torch.int8,
    trt.DataType.BOOL:  torch.bool,
}

class TrtEngine:
    def __init__(self, engine_path: Path):
        rt = trt.Runtime(TRT_LOGGER)
        self.engine = rt.deserialize_cuda_engine(Path(engine_path).read_bytes())
        self.ctx = self.engine.create_execution_context()
        self.in_name = self.out_name = None
        for i in range(self.engine.num_io_tensors):
            n = self.engine.get_tensor_name(i)
            if self.engine.get_tensor_mode(n) == trt.TensorIOMode.INPUT:
                self.in_name = n
            else:
                self.out_name = n
        self.in_dt    = _TRT2T[self.engine.get_tensor_dtype(self.in_name)]
        self.out_dt   = _TRT2T[self.engine.get_tensor_dtype(self.out_name)]
        self.in_shape  = tuple(self.engine.get_tensor_shape(self.in_name))
        self.out_shape = tuple(self.engine.get_tensor_shape(self.out_name))

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(self.in_dt).contiguous().cuda()
        out = torch.empty(self.out_shape, dtype=self.out_dt, device='cuda')
        self.ctx.set_tensor_address(self.in_name, x.data_ptr())
        self.ctx.set_tensor_address(self.out_name, out.data_ptr())
        stream = torch.cuda.current_stream()
        self.ctx.execute_async_v3(stream_handle=stream.cuda_stream)
        stream.synchronize()
        return out

def onnx_input_shape(path: Path):
    m = onnx.load(str(path))
    d = m.graph.input[0].type.tensor_type.shape.dim
    return tuple(x.dim_value for x in d)

def run_onnx(path: Path, x: torch.Tensor) -> torch.Tensor:
    sess = ort.InferenceSession(str(path), providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
    in_name = sess.get_inputs()[0].name
    y = sess.run(None, {in_name: x.detach().cpu().numpy()})[0]
    return torch.from_numpy(y).to(device)

def to_tokens(out: torch.Tensor) -> torch.Tensor:
    """Encoder output -> (B, H', W') int64 tokens. Handles both shapes:
    the shipped DALL-E encoder.pkl returns (B, H', W') tokens already (argmax
    is baked in); the exported `enc.onnx` returns (B, V, H', W') logits."""
    if out.ndim == 3:
        return out.long()
    return out.argmax(dim=1).long()

In [ ]:
# Load PyTorch reference encoder + the deployment decoder.
# The deployment pipeline (codeword_subscriber.py) uses original DALL-E encoder.pkl
# + decoder.pkl as the reference pair, so that is the reference here.
ref_enc = load_model('https://cdn.openai.com/dall-e/encoder.pkl', device).eval()
ref_dec = load_model('https://cdn.openai.com/dall-e/decoder.pkl', device).eval()
for q in ref_enc.parameters():
    q.requires_grad_(False)
for q in ref_dec.parameters():
    q.requires_grad_(False)
print('reference encoder + decoder loaded')

In [5]:
# Reference reconstruction quality: PT DALL-E encoder -> PT DALL-E decoder.
# This is the upper bound; anything we export should land near here.
def reference_recon(H: int, W: int, n: int = N_EVAL):
    xs = load_test_batch(n, H, W)
    xs_in = map_pixels(xs)
    tokens = ref_enc(xs_in).long()  # (B, H', W')
    oh = F.one_hot(tokens, num_classes=VOCAB).permute(0, 3, 1, 2).float()
    stats = ref_dec(oh).float()
    rec = unmap_pixels(torch.sigmoid(stats[:, :3])).clamp(0, 1)
    mse, psnr = mse_psnr(xs, rec)
    return xs, tokens, rec, (mse, psnr)

# Pre-compute references at both standard resolutions so we can compare quickly.
REFS = {}
for HW in [(240, 320), (480, 640)]:
    H, W = HW
    xs, tokens, rec, (mse, psnr) = reference_recon(H, W)
    REFS[HW] = dict(xs=xs, tokens=tokens, rec=rec, mse=mse, psnr=psnr)
    print(f'reference {H}x{W}: PT-enc -> PT-dec  PSNR = {psnr:5.2f} dB  MSE = {mse:.6f}')

/home/leonard/.local/lib/python3.10/site-packages/dall_e/utils.py:43: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:919.)
  return F.conv2d(x, w, b, padding=(self.kw - 1) // 2)


reference 240x320: PT-enc -> PT-dec  PSNR = 25.98 dB  MSE = 0.002524
reference 480x640: PT-enc -> PT-dec  PSNR = 25.93 dB  MSE = 0.002555


In [ ]:
def verify_encoder(path: str):
    """Run one exported encoder artifact end-to-end and print a report."""
    print(f'\n=== {path} ===')
    p = Path(path)
    if not p.exists():
        print('  MISSING'); return

    is_onnx = p.suffix == '.onnx'
    if is_onnx:
        in_shape = onnx_input_shape(p)
        runner = lambda x: run_onnx(p, x)
        in_dt_label = 'fp32'
    else:
        eng = TrtEngine(p)
        in_shape = eng.in_shape
        runner = eng
        in_dt_label = str(eng.in_dt).split('.')[-1]
        print(f'  TRT in dtype={eng.in_dt}  out dtype={eng.out_dt}  out shape={eng.out_shape}')

    if len(in_shape) != 4 or in_shape[1] != 3:
        print(f'  unexpected input shape {in_shape}'); return
    B, _, H, W = in_shape
    print(f'  in shape={in_shape}  ({in_dt_label})')
    if (H, W) not in REFS:
        xs, tokens_ref, rec_ref, (mse_ref, psnr_ref) = reference_recon(H, W)
        REFS[(H, W)] = dict(xs=xs, tokens=tokens_ref, rec=rec_ref, mse=mse_ref, psnr=psnr_ref)
    ref = REFS[(H, W)]

    xs    = ref['xs'][:N_EVAL]
    xs_in = map_pixels(xs)

    # Engines are typically built with fixed batch size B. Run one image at a time.
    tok_chunks = []
    for i in range(xs_in.shape[0]):
        out = runner(xs_in[i:i+1])
        tok_chunks.append(to_tokens(out))
    tokens = torch.cat(tok_chunks, dim=0)  # (N, H', W')
    print(f'  tokens shape={tuple(tokens.shape)}  vocab range=[{tokens.min().item()}, {tokens.max().item()}]')

    # Token agreement vs PT DALL-E (the deployment reference).
    agree_dalle = (tokens == ref['tokens'][:N_EVAL]).float().mean().item()
    print(f'  token agreement vs PT DALL-E enc:  {agree_dalle * 100:5.1f}%')

    # End-to-end: feed tokens to deployment decoder (PT DALL-E decoder.pkl).
    oh = F.one_hot(tokens, num_classes=VOCAB).permute(0, 3, 1, 2).float()
    stats = ref_dec(oh).float()
    rec = unmap_pixels(torch.sigmoid(stats[:, :3])).clamp(0, 1)
    mse, psnr = mse_psnr(xs, rec)
    # Drift relative to PT-PT reference reconstruction.
    mse_d, psnr_d = mse_psnr(rec, ref['rec'][:N_EVAL])
    print(f'  recon vs original:                  MSE={mse:.6f}  PSNR={psnr:5.2f} dB '
          f'(ref {ref["psnr"]:5.2f} dB, drop {ref["psnr"] - psnr:+5.2f})')
    print(f'  recon vs PT-enc reference recon:    MSE={mse_d:.6e}  PSNR={psnr_d:5.2f} dB')
    return dict(path=path, in_shape=in_shape, agree=agree_dalle, mse=mse, psnr=psnr,
                rec=rec, xs=xs)

In [7]:
results = []
for path in ENC_CANDIDATES:
    r = verify_encoder(path)
    if r is not None:
        results.append(r)

print('\n' + '=' * 78)
print(f'{"artifact":<55} {"agree%":>8} {"PSNR":>8}')
print('=' * 78)
for r in results:
    print(f'{Path(r["path"]).name:<55} {r["agree"] * 100:7.1f}% {r["psnr"]:7.2f}')
for HW, ref in REFS.items():
    print(f'{f"(reference PT-enc->PT-dec {HW[0]}x{HW[1]})":<55} {100.0:7.1f}% {ref["psnr"]:7.2f}')


=== /home/leonard/arc_ws/src/enc.trt ===
  TRT in dtype=torch.float32  out dtype=torch.int32  out shape=(1, 60, 80)
  in shape=(1, 3, 480, 640)  (float32)
  tokens shape=(8, 60, 80)  vocab range=[0, 8191]
  token agreement vs PT DALL-E enc:   92.0%
  token agreement vs trained ArcEncoder (btn64):  45.2%
  recon vs original:                  MSE=0.002557  PSNR=25.92 dB (ref 25.93 dB, drop +0.00)
  recon vs PT-enc reference recon:    MSE=7.285061e-05  PSNR=41.38 dB

=== /home/leonard/arc_ws/src/enc.dense.trt ===
  TRT in dtype=torch.float32  out dtype=torch.float16  out shape=(1, 8192, 60, 80)
  in shape=(1, 3, 480, 640)  (float32)
  tokens shape=(8, 60, 80)  vocab range=[4, 8190]
  token agreement vs PT DALL-E enc:    1.9%
  token agreement vs trained ArcEncoder (btn64):   1.4%
  recon vs original:                  MSE=0.175157  PSNR= 7.57 dB (ref 25.93 dB, drop +18.36)
  recon vs PT-enc reference recon:    MSE=1.740829e-01  PSNR= 7.59 dB

=== /home/leonard/arc_ws/src/enc.sparse.trt ==

2026-05-18 15:12:50.271537455 [E:onnxruntime:Default, provider_bridge_ort.cc:1992 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1637 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library libonnxruntime_providers_cuda.so with error: libcudnn.so.9: cannot open shared object file: No such file or directory

2026-05-18 15:12:50.271694883 [W:onnxruntime:Default, onnxruntime_pybind_state.cc:965 CreateExecutionProviderInstance] Failed to create CUDAExecutionProvider. Require cuDNN 9.* and CUDA 12.*. Please install all dependencies as mentioned in the GPU requirements page (https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html#requirements), make sure they're in the PATH, and that your GPU is supported.
2026-05-18 15:12:53.722526366 [E:onnxruntime:Default, provider_bridge_ort.cc:1992 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.

KeyboardInterrupt: 

In [ ]:
# Side-by-side reconstructions for a quick visual gut-check.
# Each row = one test image; columns = [original, PT-PT reference] + each engine.
import matplotlib.pyplot as plt

N_SHOW = min(4, N_EVAL)
# Group results by resolution so the grid stays consistent.
by_hw = {}
for r in results:
    by_hw.setdefault((r['in_shape'][2], r['in_shape'][3]), []).append(r)

for HW, rs in by_hw.items():
    ref = REFS[HW]
    xs = ref['xs'][:N_SHOW]
    rec_ref = ref['rec'][:N_SHOW]
    cols = 2 + len(rs)
    fig, ax = plt.subplots(N_SHOW, cols, figsize=(2.5 * cols, 2.5 * N_SHOW))
    if N_SHOW == 1:
        ax = ax[None, :]
    for i in range(N_SHOW):
        ax[i, 0].imshow(xs[i].permute(1, 2, 0).cpu().numpy())
        ax[i, 0].set_title('original' if i == 0 else '')
        ax[i, 1].imshow(rec_ref[i].permute(1, 2, 0).cpu().numpy())
        ax[i, 1].set_title(f'PT-ref ({ref["psnr"]:.1f} dB)' if i == 0 else '')
        for c, r in enumerate(rs, start=2):
            ax[i, c].imshow(r['rec'][i].permute(1, 2, 0).cpu().numpy())
            if i == 0:
                ax[i, c].set_title(f'{Path(r["path"]).name}\n({r["psnr"]:.1f} dB)', fontsize=8)
        for c in range(cols):
            ax[i, c].axis('off')
    fig.suptitle(f'{HW[0]}x{HW[1]} reconstructions')
    fig.tight_layout()
    plt.show()